In [1]:
import torch 
import torch.nn as nn

In [2]:
class NoisyTopKGating(nn.Module):
    def __init__(self, D, N, K):
        super(NoisyTopKGating, self).__init__()
        self.W_G = torch.nn.Parameter(torch.randn((D, N)))
        self.W_N = torch.nn.Parameter(torch.randn((D, N)))
        self.normal_dist = torch.distributions.Normal(loc=0, scale=1)
        self.softplus = nn.Softplus()

        self.D = D
        self.N = N 
        self.K = K


    def forward(self, X:torch.tensor): # (B, S, D)
        (B, S, D) = X.shape
        K, N = self.K, self.N
        W_G = X @ self.W_G # (B, S, D) @ (D, N) = (B, S, N)
        W_N = X @ self.W_N

        assert W_G.shape == (B, S, self.N)
        
        e = self.normal_dist.sample((B, S, self.N)) 

        H = W_G + e * self.softplus(W_N) # (B, S, N)
        KV, KI = torch.topk(H, k=self.K, dim=-1) # (B, S, K)

        assert KI.shape == (B, S, self.K)
        assert KV.shape == (B, S, self.K)


        G = nn.Softmax(dim=-1)(KV)

        assert G.shape == (B, S, K)

        # construct G_N (B, S, N) from G (B, S, K) and KI (B, S, N) where KI are indices. Torch.scatter will help here. 
        G_N = torch.zeros((B, S, N), dtype=G.dtype)
        G_N.scatter(dim=2, index=KI, src=G)

        assert G_N.shape == (B, S, N)

        probs = G_N.mean(dim=(0, 1))

        assert probs.shape == (self.N, )

        threshold_logit = KV[:,:,-1:]

        assert threshold_logit.shape == (B, S, 1)

        D = (W_G - threshold_logit) / W_N 

        assert D.shape == (B, S, self.N)

        f = self.normal_dist.cdf(D).mean(dim=(0, 1))

        assert f.shape == (self.N, )

        aux_loss = torch.sum(f * probs)

        return G, aux_loss, KI     



        



noisy_top_k_gating = NoisyTopKGating(D=2, N=5, K=3)

X = torch.randn((1, 4, 2)).float()
noisy_top_k_gating.forward(X)

(tensor([[[0.4704, 0.4179, 0.1117],
          [0.6637, 0.2011, 0.1352],
          [0.5462, 0.2273, 0.2265],
          [0.8484, 0.1078, 0.0438]]], grad_fn=<SoftmaxBackward0>),
 tensor(0., grad_fn=<SumBackward0>),
 tensor([[[1, 0, 3],
          [1, 4, 0],
          [1, 3, 2],
          [3, 0, 1]]]))

In [15]:
a = torch.randn((1, 1, 4, 3))
print(a)
print(a.shape)

b = torch.tensor([
    [1, 2],
    [2, 0],
    [0, 1],
    [1, 0]
    ]).unsqueeze(0).unsqueeze(0)

print(b.shape)
c = torch.gather(a, dim=2, index=b)

print(c)
print(c.shape)

tensor([[[[ 1.3092, -1.5253, -0.3377],
          [-0.5952,  0.8542,  0.5866],
          [ 0.5613,  0.3461,  0.3864],
          [ 1.1665,  0.1827,  0.1150]]]])
torch.Size([1, 1, 4, 3])
torch.Size([1, 1, 4, 2])
tensor([[[[-0.5952,  0.3461],
          [ 0.5613, -1.5253],
          [ 1.3092,  0.8542],
          [-0.5952, -1.5253]]]])
torch.Size([1, 1, 4, 2])


In [21]:
src = torch.arange(1, 11).reshape((2, 5))
print(src)

index = torch.tensor([[0, 1, 2, 0]])

print(index, index.shape)

torch.zeros((3, 5), dtype=src.dtype).scatter_(0, index, src)

tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10]])
tensor([[0, 1, 2, 0]]) torch.Size([1, 4])


tensor([[1, 0, 0, 4, 0],
        [0, 2, 0, 0, 0],
        [0, 0, 3, 0, 0]])

In [24]:
class Block(nn.Module):
    def __init__(self, D):
        super(Block, self).__init__()
        self.a1 = nn.Linear(D, D)
        self.a2 = nn.Linear(D, D)

    def forward(self, x):
        h1 = self.a1(x)
        h2 = nn.GELU()(x)
        return self.a2(h2)

class ShazeerMOE(nn.Module):
    def __init__(self, D, N, K):
        super(ShazeerMOE, self).__init__()
        self.D = D
        self.N = N 
        self.K = K
        
        self.experts = []
        for _ in range(N):
            new_block = Block(D)
            self.experts.append(new_block)

        self.noisy_gating = NoisyTopKGating(D, N, K)

    def forward(self, X):
        B, S, D = X.shape
        K, N = self.K, self.N

        G, aux_loss, KI = self.noisy_gating(X)

        assert KI.shape == (B, S, K)

        XE = torch.zeros((N, B, S, D))
        for i in range(len(self.experts)):
            XE[i] = self.experts[i].forward((X))

        XE = XE.permute(1, 2, 3, 0)

        assert XE.shape == (B, S, D, N)

        KID = KI.unsqueeze(dim=2).expand(-1, -1, D, -1)

        assert KID.shape == (B, S, D, K)


        XEK = torch.gather(XE, dim=-1, index=KID)

        assert XEK.shape == (B, S, D, K)
        
        y =  torch.einsum("bsdk,bsk->bsd", XEK, G)

        return y, aux_loss

D = 2
N = 4
K = 2
B = 1
S = 3
shazeer_moe = ShazeerMOE(D, N, K)
a = torch.randn((B, S, D))
shazeer_moe.forward(a)


(tensor([[[ 0.0372,  0.2457],
          [ 0.1832,  0.2329],
          [-0.1843, -0.0899]]], grad_fn=<ViewBackward0>),
 tensor(0., grad_fn=<SumBackward0>))

In [8]:
dim=2, dim=2, 

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (1058529825.py, line 1)

In [ ]:
src = torch.arange(1, 11).reshape((2, 5))
src
index = torch.tensor([[0, 1, 2, 0]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(0, index, src)
index = torch.tensor([[0, 1, 2], [0, 1, 4]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(1, index, src)

torch.full((2, 4), 2.).scatter_(1, torch.tensor([[2], [3]]),
           1.23, reduce='multiply')
torch.full((2, 4), 2.).scatter_(1, torch.tensor([[2], [3]]),
           1.23, reduce='add')src = torch.arange(1, 11).reshape((2, 5))
src
index = torch.tensor([[0, 1, 2, 0]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(0, index, src)
index = torch.tensor([[0, 1, 2], [0, 1, 4]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(1, index, src)

torch.full((2, 4), 2.).scatter_(1, torch.tensor([[2], [3]]),
           1.23, reduce='multiply')
torch.full((2, 4), 2.).scatter_(1, torch.tensor([[2], [3]]),
           1.23, reduce='add')src = torch.arange(1, 11).reshape((2, 5))
src
index = torch.tensor([[0, 1, 2, 0]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(0, index, src)
index = torch.tensor([[0, 1, 2], [0, 1, 4]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(1, index, src)

torch.full((2, 4), 2.).scatter_(1, torch.tensor([[2], [3]]),
           1.23, reduce='multiply')
torch.full((2, 4), 2.).scatter_(1, torch.tensor([[2], [3]]),
           1.23, reduce='add')src = torch.arange(1, 11).reshape((2, 5))
           1.23, reduce='add')src = torch.arange(1, 11).reshape((2, 5))s